# Comprimere e ricostruire, e perché non basta

Il codice del capitolo [«Comprimere e ricostruire, e perché non basta»](https://book.paithon.it/main/ModelliLatenti/comprimere-e-ricostruire.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q scikit-learn torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Comprimere e ricostruire, e perché non basta

[Leggi la pagina](https://book.paithon.it/main/ModelliLatenti/comprimere-e-ricostruire.html)


### Trenta righe, e funziona


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.datasets import load_digits

torch.manual_seed(0)
# un thread solo: cosi' i numeri stampati qui sotto sono gli stessi su
# qualunque macchina, e su dati piccoli come questi e' anche piu' veloce
torch.set_num_threads(1)

# 1797 cifre scritte a mano, 8x8 pixel, riportate fra 0 (chiaro) e 1 (scuro)
X = torch.tensor(load_digits().data / 16.0, dtype=torch.float32)


class Clessidra(nn.Module):
    """Encoder e decoder, con la strozzatura in mezzo."""

    def __init__(self, latente=8):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 48), nn.ReLU(),
                                     nn.Linear(48, latente))
        self.decoder = nn.Sequential(nn.Linear(latente, 48), nn.ReLU(),
                                     nn.Linear(48, 64))


rete = Clessidra()
opt = torch.optim.Adam(rete.parameters(), lr=3e-3)
for passo in range(4000):
    # il decoder esce in logit; la cross-entropia li confronta col grigio vero
    perdita = F.binary_cross_entropy_with_logits(
        rete.decoder(rete.encoder(X)), X, reduction="sum") / len(X)
    opt.zero_grad()
    perdita.backward()
    opt.step()

print(f"errore di ricostruzione:   {perdita.item():.1f} nat per cifra")

# il metro di paragone: chi non guarda la cifra e dichiara, per ogni pixel,
# il grigio medio che quel pixel ha su tutte le 1797 cifre
marginale = F.binary_cross_entropy(
    X.mean(0).expand_as(X), X, reduction="sum") / len(X)
print(f"chi non guarda la cifra:  {marginale.item():.1f} nat per cifra")

In [ ]:
LIVELLI = " .:-=+*#%"


def affianca(*immagini):
    """Le immagini 8x8 stampate una accanto all'altra, in caratteri."""
    griglie = [(im.reshape(8, 8) * 8).round().long().clamp(0, 8) for im in immagini]
    return "\n".join("   ".join("".join(LIVELLI[i] for i in g[r]) for g in griglie)
                     for r in range(8))


with torch.no_grad():
    codici = rete.encoder(X)
    ricostruite = torch.sigmoid(rete.decoder(codici))

print("quattro cifre vere")
print(affianca(*X[:4]))
print("\nle stesse, rifatte a partire da otto numeri")
print(affianca(*ricostruite[:4]))

### Il cammino che si perde


In [ ]:
with torch.no_grad():
    partenza, arrivo = codici[0], codici[1]      # lo zero e l'uno di prima
    tappe = torch.stack([partenza + t * (arrivo - partenza)
                         for t in torch.linspace(0, 1, 5)])
    print(affianca(*torch.sigmoid(rete.decoder(tappe))))

In [ ]:
with torch.no_grad():
    sorteggiati = codici.mean(0) + codici.std(0) * torch.randn(500, 8)
    inventate = torch.sigmoid(rete.decoder(sorteggiati))

print("quattro cifre decodificate da codici sorteggiati")
print(affianca(*inventate[:4]))

fra_codici = torch.cdist(codici, codici)
fra_codici.fill_diagonal_(float("inf"))
spaziatura = fra_codici.min(1).values.median()
lontananza = torch.cdist(sorteggiati, codici).min(1).values.median()

print(f"\nfra un codice vero e il suo vicino:   {spaziatura:.2f}")
print(f"fra un codice sorteggiato e i veri:   {lontananza:.2f}"
      f"   ({lontananza / spaziatura:.1f} volte la spaziatura)")

## Il salto probabilistico: l’ELBO e la riparametrizzazione

[Leggi la pagina](https://book.paithon.it/main/ModelliLatenti/il-salto-probabilistico.html)


### Il conto che non si può fare


In [ ]:
import math
import torch

torch.manual_seed(0)
torch.set_num_threads(1)      # numeri riproducibili su qualunque macchina

# modello giocattolo: z ~ N(0, I), x|z ~ N(z, sigma^2 I). Qui p(x) si sa:
# marginalizzando due gaussiane ne esce una sola, N(0, (1 + sigma^2) I).
SIGMA, CAMPIONI = 0.5, 100_000


def log_p_vero(x):
    var = 1 + SIGMA ** 2
    return (-0.5 * (x ** 2).sum() / var
            - 0.5 * len(x) * math.log(2 * math.pi * var)).item()


def log_p_stimato(x, campioni=CAMPIONI):
    """La media di p(x|z) su z sorteggiati dal prior, in scala logaritmica."""
    z = torch.randn(campioni, len(x))
    log_p_x_dato_z = (-0.5 * ((x - z) ** 2).sum(1) / SIGMA ** 2
                      - 0.5 * len(x) * math.log(2 * math.pi * SIGMA ** 2))
    return torch.logsumexp(log_p_x_dato_z, 0).item() - math.log(campioni), log_p_x_dato_z


print(f"{'dimensioni':>10} {'log p(x) vero':>14} {'stimato':>10} "
      f"{'errore':>8} {'peso del piu grosso':>21}")
for L in (1, 2, 5, 10, 20, 40):
    x = torch.full((L,), 0.6)          # un dato qualunque, lo stesso in ogni dimensione
    stima, pesi = log_p_stimato(x)
    quota = (pesi.max() - torch.logsumexp(pesi, 0)).exp().item()
    print(f"{L:>10} {log_p_vero(x):>14.2f} {stima:>10.2f} "
          f"{stima - log_p_vero(x):>8.2f} {quota:>20.1%}")

### Il trucco della riparametrizzazione


In [ ]:
import torch

torch.manual_seed(0)

# Vogliamo la derivata rispetto a mu di E[z^2] con z ~ N(mu, 1).
# Il valore vero si sa: E[z^2] = mu^2 + 1, quindi la derivata e' 2*mu.
MU, PROVE = 2.0, 200_000

epsilon = torch.randn(PROVE)
z = MU + epsilon                      # gli stessi sorteggi per i due metodi

# 1. si deriva attraverso il sorteggio: z e' mu piu' rumore, quindi d(z^2)/dmu = 2z
riparametrizzato = 2 * z

# 2. non si deriva il sorteggio, si deriva la probabilita' di averlo pescato:
#    per una gaussiana, d(log q)/dmu = z - mu
punteggio = z ** 2 * (z - MU)

for nome, stima in (("riparametrizzazione", riparametrizzato), ("punteggio", punteggio)):
    print(f"{nome:>20}: media {stima.mean():6.3f}   "
          f"deviazione standard {stima.std():7.3f}")
print(f"{'valore vero':>20}: {2 * MU:6.3f}")
print(f"\nla varianza del secondo e' {(punteggio.var() / riparametrizzato.var()):.0f} "
      f"volte quella del primo")

### Tutto insieme


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.datasets import load_digits

torch.manual_seed(0)
torch.set_num_threads(1)
X = torch.tensor(load_digits().data / 16.0, dtype=torch.float32)
LATENTE = 8


class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.tronco = nn.Sequential(nn.Linear(64, 48), nn.ReLU())
        self.testa = nn.Linear(48, 2 * LATENTE)   # media e log-varianza insieme
        self.decoder = nn.Sequential(nn.Linear(LATENTE, 48), nn.ReLU(),
                                     nn.Linear(48, 64))

    def codifica(self, x):
        return self.testa(self.tronco(x)).chunk(2, dim=1)


vae = VAE()
opt = torch.optim.Adam(vae.parameters(), lr=3e-3)
for passo in range(4000):
    media, log_var = vae.codifica(X)
    # il sorteggio scritto in modo derivabile: rumore fisso, media e scala apprese
    z = media + torch.exp(0.5 * log_var) * torch.randn_like(media)

    ricostruzione = F.binary_cross_entropy_with_logits(
        vae.decoder(z), X, reduction="sum") / len(X)
    costo_descrizione = (-0.5 * (1 + log_var - media ** 2
                                 - log_var.exp()).sum(1)).mean()
    perdita = ricostruzione + costo_descrizione     # cioe' -ELBO

    opt.zero_grad()
    perdita.backward()
    opt.step()

print(f"ricostruzione      {ricostruzione.item():6.1f} nat")
print(f"costo descrizione  {costo_descrizione.item():6.1f} nat")
print(f"ELBO              {-perdita.item():7.1f} nat  "
      f"(log p(x) sta piu' in alto di qui)")

In [ ]:
LIVELLI = " .:-=+*#%"


def affianca(*immagini):
    griglie = [(im.reshape(8, 8) * 8).round().long().clamp(0, 8) for im in immagini]
    return "\n".join("   ".join("".join(LIVELLI[i] for i in g[r]) for g in griglie)
                     for r in range(8))


with torch.no_grad():
    # si pesca dal vocabolario comune, che nel codice si chiama prior
    nuove = torch.sigmoid(vae.decoder(torch.randn(500, LATENTE)))

print("quattro cifre pescate dal prior e decodificate")
print(affianca(*nuove[:4]))

In [ ]:
class Clessidra(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 48), nn.ReLU(),
                                     nn.Linear(48, LATENTE))
        self.decoder = nn.Sequential(nn.Linear(LATENTE, 48), nn.ReLU(),
                                     nn.Linear(48, 64))


torch.manual_seed(0)
ae = Clessidra()
opt = torch.optim.Adam(ae.parameters(), lr=3e-3)
for passo in range(4000):
    perdita_ae = F.binary_cross_entropy_with_logits(
        ae.decoder(ae.encoder(X)), X, reduction="sum") / len(X)
    opt.zero_grad()
    perdita_ae.backward()
    opt.step()


def quanto_e_vuoto(visti, sorteggiati):
    """Di quante spaziature tipiche un codice sorteggiato manca il bersaglio."""
    fra = torch.cdist(visti, visti)
    fra.fill_diagonal_(float("inf"))
    return (torch.cdist(sorteggiati, visti).min(1).values.median()
            / fra.min(1).values.median()).item()


def quanto_somiglia(immagini, veri):
    return torch.cdist(immagini, veri).min(1).values.median().item()


with torch.no_grad():
    codici_ae = ae.encoder(X)
    sorteggiati_ae = codici_ae.mean(0) + codici_ae.std(0) * torch.randn(500, LATENTE)
    media, log_var = vae.codifica(X)
    codici_vae = media + torch.exp(0.5 * log_var) * torch.randn_like(media)
    sorteggiati_vae = torch.randn(500, LATENTE)
    nuove_ae = torch.sigmoid(ae.decoder(sorteggiati_ae))

fra_veri = torch.cdist(X, X)
fra_veri.fill_diagonal_(float("inf"))
print(f"{'':<26}{'buchi nel latente':>18}{'distanza dalle vere':>22}")
print(f"{'cifra vera':<26}{'':>18}{fra_veri.min(1).values.median():>22.2f}")
print(f"{'autoencoder':<26}{quanto_e_vuoto(codici_ae, sorteggiati_ae):>17.1f}x"
      f"{quanto_somiglia(nuove_ae, X):>22.2f}")
print(f"{'autoencoder variazionale':<26}{quanto_e_vuoto(codici_vae, sorteggiati_vae):>17.1f}x"
      f"{quanto_somiglia(nuove, X):>22.2f}")

## Il latente che si usa

[Leggi la pagina](https://book.paithon.it/main/ModelliLatenti/il-latente-che-si-usa.html)


### Una manopola sul costo della scheda


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.datasets import load_digits

torch.set_num_threads(1)      # numeri riproducibili su qualunque macchina
X = torch.tensor(load_digits().data / 16.0, dtype=torch.float32)
LATENTE = 8


class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.tronco = nn.Sequential(nn.Linear(64, 48), nn.ReLU())
        self.testa = nn.Linear(48, 2 * LATENTE)
        self.decoder = nn.Sequential(nn.Linear(LATENTE, 48), nn.ReLU(),
                                     nn.Linear(48, 64))

    def codifica(self, x):
        return self.testa(self.tronco(x)).chunk(2, dim=1)


def addestra(beta):
    """Lo stesso VAE della sezione scorsa, col costo di descrizione pesato."""
    torch.manual_seed(0)
    vae = VAE()
    opt = torch.optim.Adam(vae.parameters(), lr=3e-3)
    for passo in range(4000):
        media, log_var = vae.codifica(X)
        z = media + torch.exp(0.5 * log_var) * torch.randn_like(media)
        ricostruzione = F.binary_cross_entropy_with_logits(
            vae.decoder(z), X, reduction="sum") / len(X)
        # il costo tenuto separato riga per riga, per poterlo poi leggere
        costo = (-0.5 * (1 + log_var - media ** 2 - log_var.exp())).mean(0)
        perdita = ricostruzione + beta * costo.sum()
        opt.zero_grad()
        perdita.backward()
        opt.step()
    return vae, ricostruzione.item(), costo.detach()


print(f"{'beta':>5} {'ricostruzione':>14} {'costo':>7} {'righe usate':>12}   nat per riga")
reti = {}
for beta in (0.5, 1, 2, 4):
    reti[beta], ricostruzione, costo = addestra(beta)
    print(f"{beta:>5} {ricostruzione:>14.1f} {costo.sum():>7.2f} "
          f"{(costo > 0.05).sum().item():>9}/{LATENTE}   "
          + " ".join(f"{v:.2f}" for v in costo.sort(descending=True).values))

In [ ]:
LIVELLI = " .:-=+*#%"


def affianca(*immagini):
    griglie = [(im.reshape(8, 8) * 8).round().long().clamp(0, 8) for im in immagini]
    return "\n".join("   ".join("".join(LIVELLI[i] for i in g[r]) for g in griglie)
                     for r in range(8))


for beta in (1, 4):
    with torch.no_grad():
        vae = reti[beta]
        media, log_var = vae.codifica(X)
        costo = (-0.5 * (1 + log_var - media ** 2 - log_var.exp())).mean(0)
        riga = int(costo.argmax())
        varianti = media[:1].repeat(5, 1)          # la scheda della prima cifra
        varianti[:, riga] = torch.linspace(-2.5, 2.5, 5)
        print(f"\nbeta = {beta}: la riga {riga}, la piu' carica, "
              f"portata da -2,5 a +2,5")
        print(affianca(*torch.sigmoid(vae.decoder(varianti))))